In [ ]:
# IMPORTS

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

In [ ]:
# PATH

file_path = "/kaggle/input/datasets/siddhuuu101/topgun1/top_gun_opendata_1.parquet"

In [ ]:
# SPLIT

from sklearn.model_selection import train_test_split

meta = pd.read_parquet(file_path, columns=["m", "pt", "iphi", "ieta"])

bins = pd.qcut(meta["m"], q=20, labels=False, duplicates="drop")
idx = np.arange(len(meta))

train_idx, temp_idx = train_test_split(
    idx, test_size=0.20, random_state=42, stratify=bins
)

temp_bins = pd.qcut(meta.iloc[temp_idx]["m"], q=10, labels=False, duplicates="drop")

val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=42, stratify=temp_bins
)

print("Train:", len(train_idx))
print("Val:", len(val_idx))
print("Test:", len(test_idx))

# NORMALIZATION (X + Y)

pf = pq.ParquetFile(file_path)

sample_idx = np.random.choice(train_idx, 2000, replace=False)

all_vals = []
m_vals = []

for i in sample_idx:
    row = pf.read_row_groups([int(i)]).to_pandas().iloc[0]

    m_vals.append(row['m'])

    channels = []
    for ch in row['X_jet']:
        ch_arr = np.array(ch, dtype=object)
        ch_arr = np.array(ch_arr.tolist(), dtype=np.float32).reshape(-1)
        channels.append(ch_arr)

    arr = np.stack(channels)
    arr = np.sign(arr) * np.log1p(np.abs(arr))

    all_vals.append(arr.reshape(-1))

all_vals = np.concatenate(all_vals)

x_mean = all_vals.mean()
x_std  = all_vals.std()

y_mean = np.mean(m_vals)
y_std  = np.std(m_vals)

print("X mean/std:", x_mean, x_std)
print("Y mean/std:", y_mean, y_std)

# AUX NORMALIZATION 

aux_vals = []

aux_sample_idx = np.random.choice(train_idx, 2000, replace=False)

for i in aux_sample_idx:
    row = pf.read_row_groups([int(i)]).to_pandas().iloc[0]
    aux_vals.append([row['pt'], row['iphi'], row['ieta']])

aux_vals = np.array(aux_vals)

aux_mean = aux_vals.mean(axis=0)
aux_std  = aux_vals.std(axis=0)

print("Aux mean:", aux_mean)
print("Aux std:", aux_std)

# DATASET

class JetDataset(Dataset):
    def __init__(self, parquet_file, indices,
                 x_mean, x_std, y_mean, y_std,
                 aux_mean, aux_std):

        self.pf = pq.ParquetFile(parquet_file)
        self.indices = indices

        self.x_mean = x_mean
        self.x_std = x_std
        self.y_mean = y_mean
        self.y_std = y_std

        self.aux_mean = aux_mean
        self.aux_std = aux_std

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = int(self.indices[idx])
        row = self.pf.read_row_groups([real_idx]).to_pandas().iloc[0]

        # -------- IMAGE --------
        sample = row['X_jet']

        channels = []
        for ch in sample:
            ch_arr = np.array(ch, dtype=object)
            ch_arr = np.array(ch_arr.tolist(), dtype=np.float32).reshape(-1)
            channels.append(ch_arr)

        arr = np.stack(channels)

        x = arr[:4].reshape(4, 125, 125)

        x = np.sign(x) * np.log1p(np.abs(x))
        x = (x - self.x_mean) / (self.x_std + 1e-6)

        # -------- AUX --------
        aux = np.array([row['pt'], row['iphi'], row['ieta']], dtype=np.float32)
        aux = (aux - self.aux_mean) / (self.aux_std + 1e-6)

        # -------- TARGET --------
        y = row['m']
        y = (y - self.y_mean) / (self.y_std + 1e-6)

        return (
            torch.from_numpy(x).float(),
            torch.from_numpy(aux).float(),
            torch.tensor(y, dtype=torch.float32)
        )

# DATALOADERS

train_dataset = JetDataset(
    file_path, train_idx,
    x_mean, x_std, y_mean, y_std,
    aux_mean, aux_std
)

val_dataset = JetDataset(
    file_path, val_idx,
    x_mean, x_std, y_mean, y_std,
    aux_mean, aux_std
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0)


# FINAL CHECK

for x, aux, y in train_loader:
    print(x.shape, aux.shape, y.shape)
    break

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        scale = self.se(x).view(x.size(0), x.size(1), 1, 1)
        return x * scale


class JetCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(4, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.layer1 = nn.Sequential(
            ResBlock(64),
            SEBlock(64)
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
            ResBlock(128),
            SEBlock(128)
        )

        self.layer3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.ReLU(),
            ResBlock(256),
            SEBlock(256)
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.regressor = nn.Sequential(
            nn.Linear(256 + 3, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
            
        )

    def forward(self, x, aux):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)

        x = torch.cat([x, aux], dim=1)

        return self.regressor(x).squeeze(1)

In [ ]:
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = JetCNN().to(device)

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, factor=0.5, patience=2
)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

y_mean_t = torch.tensor(y_mean, device=device)
y_std_t = torch.tensor(y_std, device=device)

In [ ]:
best_model_state = None
best_val_loss = float("inf")
patience = 2
counter = 0

epochs = 10 

for epoch in range(epochs):

    # ===== TRAIN =====
    model.train()
    train_loss = 0

    for x, aux, y in train_loader:
        x = x.to(device)
        aux = aux.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=(device.type=="cuda")):
            pred = model(x, aux)
            loss = criterion(pred, y)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    # ===== VALIDATION =====
    model.eval()
    val_loss = 0

    preds_all = []
    trues_all = []

    with torch.no_grad():
        for x, aux, y in val_loader:
            x = x.to(device)
            aux = aux.to(device)
            y = y.to(device)

            pred = model(x, aux)
            loss = criterion(pred, y)

            val_loss += loss.item()

            preds_all.append(pred.detach())
            trues_all.append(y.detach())

    preds = torch.cat(preds_all)
    trues = torch.cat(trues_all)

    # denormalize
    preds = preds * y_std_t + y_mean_t
    trues = trues * y_std_t + y_mean_t

    mae = torch.mean(torch.abs(preds - trues)).item()
    rmse = torch.sqrt(torch.mean((preds - trues)**2)).item()

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
       best_val_loss = val_loss
       counter = 0
       best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
       counter += 1

    if counter >= patience:
       print("Early stopping")
       break
    
    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss/len(train_loader):.4f}")
    print(f"Val Loss:   {val_loss/len(val_loader):.4f}")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")
    print("-"*40)
    

model.load_state_dict(best_model_state)
torch.save(model.state_dict(), "best_model.pth")

In [ ]:
with open("metrics.txt", "w") as f:
    f.write(f"Best MAE: {mae}\n")
    f.write(f"Best RMSE: {rmse}\n")

In [ ]:
import pickle
from sklearn.metrics import r2_score

# compute r2
r2 = r2_score(trues.cpu().numpy(), preds.cpu().numpy())

# save model
torch.save(model.state_dict(), "best_model.pth")

# save normalization
norm_dict = {
    "x_mean": x_mean,
    "x_std": x_std,
    "y_mean": y_mean,
    "y_std": y_std,
    "aux_mean": aux_mean,
    "aux_std": aux_std
}

with open("norm.pkl", "wb") as f:
    pickle.dump(norm_dict, f)

# save metrics
with open("metrics.txt", "w") as f:
    f.write(f"MAE: {mae}\n")
    f.write(f"RMSE: {rmse}\n")
    f.write(f"R2: {r2}\n")

print("Saved model + norm + metrics")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# convert tensors
preds_np = preds.cpu().numpy()
trues_np = trues.cpu().numpy()
residuals = preds_np - trues_np

# FIGURE

fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# 1. SCATTER: PRED VS TRUE

ax = axs[0, 0]
ax.scatter(trues_np, preds_np, s=5, alpha=0.3)

lo = min(trues_np.min(), preds_np.min())
hi = max(trues_np.max(), preds_np.max())

ax.plot([lo, hi], [lo, hi], 'r--')
ax.set_title(f"Prediction vs Truth\nMAE={mae:.2f}, RMSE={rmse:.2f}")
ax.set_xlabel("True Mass")
ax.set_ylabel("Predicted Mass")
ax.grid(True)

# 2. RESIDUAL HISTOGRAM

ax = axs[0, 1]
ax.hist(residuals, bins=60)
ax.axvline(0, linestyle="--")
ax.set_title("Residual Distribution")
ax.set_xlabel("Pred - True")
ax.set_ylabel("Count")
ax.grid(True)

# 3. RESIDUAL VS TRUE

ax = axs[1, 0]
ax.scatter(trues_np, residuals, s=5, alpha=0.3)
ax.axhline(0, linestyle="--")
ax.set_title("Residual vs True Mass")
ax.set_xlabel("True Mass")
ax.set_ylabel("Residual")
ax.grid(True)

# 4. DISTRIBUTION COMPARISON

ax = axs[1, 1]
ax.hist(trues_np, bins=50, alpha=0.6, label="True")
ax.hist(preds_np, bins=50, alpha=0.6, label="Predicted")
ax.set_title("Mass Distribution")
ax.set_xlabel("Mass")
ax.set_ylabel("Count")
ax.legend()
ax.grid(True)

# SAVE + SHOW

plt.tight_layout()
plt.savefig("dashboard.png", dpi=150)
plt.show()

print("Dashboard saved: dashboard.png")

In [ ]:
import torch
import pickle
import numpy as np
import pyarrow.parquet as pq

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load model
model = JetCNN().to(device)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

# load normalization
with open("norm.pkl", "rb") as f:
    norm = pickle.load(f)

x_mean = norm["x_mean"]
x_std  = norm["x_std"]
y_mean = norm["y_mean"]
y_std  = norm["y_std"]
aux_mean = norm["aux_mean"]
aux_std  = norm["aux_std"]

print("Model + normalization loaded")

In [ ]:
def predict_single(row):

    # -------- IMAGE --------
    channels = []
    for ch in row['X_jet']:
        ch_arr = np.array(ch, dtype=object)
        ch_arr = np.array(ch_arr.tolist(), dtype=np.float32).reshape(-1)
        channels.append(ch_arr)

    arr = np.stack(channels)
    x = arr[:4].reshape(4, 125, 125)

    x = np.sign(x) * np.log1p(np.abs(x))
    x = (x - x_mean) / (x_std + 1e-6)

    # -------- AUX --------
    aux = np.array([row['pt'], row['iphi'], row['ieta']], dtype=np.float32)
    aux = (aux - aux_mean) / (aux_std + 1e-6)

    # to tensor
    x = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)
    aux = torch.tensor(aux, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(x, aux)

    pred = pred.cpu().item() * y_std + y_mean
    return pred

In [ ]:
pf = pq.ParquetFile(file_path)

preds_test = []
trues_test = []

for i in test_idx:  
    row = pf.read_row_groups([int(i)]).to_pandas().iloc[0]

    pred = predict_single(row)
    true = row["m"]

    preds_test.append(pred)
    trues_test.append(true)

preds_test = np.array(preds_test)
trues_test = np.array(trues_test)

print("Inference done")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae  = mean_absolute_error(trues_test, preds_test)
rmse = np.sqrt(mean_squared_error(trues_test, preds_test))
r2   = r2_score(trues_test, preds_test)

print(f"TEST MAE:  {mae:.2f}")
print(f"TEST RMSE: {rmse:.2f}")
print(f"TEST R2:   {r2:.4f}")

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "true": trues_test,
    "pred": preds_test
})

df.to_csv("test_predictions.csv", index=False)

print("Saved: test_predictions.csv")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(trues_test, preds_test, s=5, alpha=0.3)
plt.plot([trues_test.min(), trues_test.max()],
         [trues_test.min(), trues_test.max()],
         'r--')

plt.xlabel("True")
plt.ylabel("Predicted")
plt.title("Test Predictions")
plt.show()

In [ ]:
import os
print(os.listdir())

In [ ]:
import torch

device = torch.device("cpu")

model = JetCNN()
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

#  Wrapper (safer for ONNX)
class Wrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input, aux):
        return self.model(input, aux)

model = Wrapper(model)

dummy_x = torch.randn(1, 4, 125, 125)
dummy_aux = torch.randn(1, 3)

torch.onnx.export(
    model,
    (dummy_x, dummy_aux),
    "sample.onnx",
    input_names=["input", "aux"],
    output_names=["output"],
    opset_version=11,
    dynamic_axes={
        "input": {0: "batch"},
        "aux": {0: "batch"},
        "output": {0: "batch"},
    },
    do_constant_folding=True,
    dynamo=False
)

print(" ONNX exported successfully")

In [ ]:
import os
import zipfile

folder_name = "results"
os.makedirs(folder_name, exist_ok=True)

files_to_copy = [
    "best_model.pth",
    "sample.onnx",
    "norm.pkl",
    "metrics.txt",
    "dashboard.png",
    "test_predictions.csv"
]

# move files into folder
for f in files_to_copy:
    if os.path.exists(f):
        os.rename(f, os.path.join(folder_name, f))

# zip folder
zip_path = "Final_submission.zip"

with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk(folder_name):
        for file in files:
            full_path = os.path.join(root, file)
            zipf.write(full_path)

print(" Folder zip created:", zip_path)